# MathSpan ruBERT v3 -- LLM-Augmented Training Data (E4d)

Two new experiments on top of BERT-0..BERT-3:

- **BERT-4**: train on `train_clean` (185) + `train_llm` (510) = 695 sentences, no curriculum.
- **BERT-5**: curriculum -- Stage 1 on `train_noisy` (1261 Mathematicon), Stage 2 (fresh
  optimizer, same in-memory model) on `train_clean` + `train_llm` (695).

Hypothesis: `train_llm` was generated specifically to match the test distribution
(Gosha real lectures) -- varied span lengths, discourse markers near boundaries,
single-token spans -- so it should help span-F1 where s2l/Mathematicon augmentation
(BERT-2/BERT-3) made it worse.

**Design notes / departures from the E4b notebook, per this task's constraints:**
- No `Trainer` API. Manual training loop (`model.train()`/`model.eval()`, manual
  `AdamW` + linear warmup, manual fp16 via `autocast`/`GradScaler`, manual gradient
  clipping) for full control over the two curriculum stages and optimizer resets.
- Best-checkpoint tracking is in-memory (CPU state-dict clone), same motivation as
  E4b: avoid disk checkpoint writes dominating wall-clock time.
- Stage 1 -> Stage 2 handoff for BERT-5 needs a **fresh optimizer**, not a fresh
  model. Since this is a manual loop (no Trainer per-stage object to recreate),
  that's just "build a new `AdamW` + scheduler for the same in-memory model" --
  no disk round-trip needed at all here (an improvement over E4b's Stage-1
  save-to-disk handoff, since this loop never had a Trainer object holding state
  to begin with).
- `bio_merged.tsv` is read directly; no HF `DatasetDict` export step.

**Do NOT change:** model checkpoint (`ai-forever/ruBert-base`), the eval protocol
(test = 50 Gosha sentences, val = 30 Gosha sentences for early stopping only),
batch size 16 (baseline), or seqeval strict/IOB2 span-F1.

Upload `bio_merged.tsv` when prompted in the data-loading cell below.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/linguistic project/bio_merged.tsv" "/content/bio_merged.tsv"

In [3]:
# --- Setup -------------------------------------------------------------
!pip -q install transformers datasets seqeval accelerate

import os
import gc
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
from seqeval.metrics import f1_score as seqeval_f1
from seqeval.scheme import IOB2
from sklearn.metrics import f1_score as sklearn_f1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEVICE = {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected. fp16/autocast will be skipped automatically; "
          "training will be slow. Use a T4 runtime on Colab (Runtime > Change runtime type).")


def flag(msg):
    '''Print a clearly-marked warning the user should not miss when scanning logs.'''
    print(f"\n[FLAG] {msg}\n")


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


SEEDS = [42, 123, 456]
MAX_LENGTH = 128
ACTIVE_MODEL_NAME = "ai-forever/ruBert-base"
FALLBACK_MODEL_NAME = "DeepPavlov/rubert-base-cased"
LABEL_LIST = ["O", "B-MATH", "I-MATH"]
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for i, l in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
DEVICE = cuda


## Data loading

`bio_merged.tsv` columns: `sentence_id`, `token_index`, `token`, `bio_tag`, `source`, `split`.
Group rows by `sentence_id` (sorted by `token_index`), recover one token/tag list per
sentence, and bucket by `split`. Expected sizes (per task spec) are printed and checked
against what's actually loaded -- a mismatch is flagged but does not stop execution,
since the source TSV is the ground truth at run time.

In [4]:
# --- Upload / locate bio_merged.tsv -------------------------------------
BIO_TSV_PATH = "/content/bio_merged.tsv"

if not os.path.exists(BIO_TSV_PATH):
    try:
        from google.colab import files
        print("bio_merged.tsv not found locally -- please upload it now.")
        uploaded = files.upload()
        # Use whatever was uploaded if the exact name differs.
        candidates = [f for f in uploaded if f.endswith(".tsv")]
        assert candidates, "No .tsv file was uploaded."
        BIO_TSV_PATH = candidates[0]
    except ImportError:
        raise FileNotFoundError(
            "bio_merged.tsv not found and google.colab is not available to prompt an "
            "upload. Place bio_merged.tsv in the working directory and re-run this cell."
        )

df = pd.read_csv(BIO_TSV_PATH, sep="\t", quoting=3, encoding="utf-8", dtype=str, keep_default_na=False)
required_cols = {"sentence_id", "token_index", "token", "bio_tag", "source", "split"}
missing = required_cols - set(df.columns)
assert not missing, f"bio_merged.tsv is missing required columns: {missing}"

df["token_index"] = df["token_index"].astype(int)
print(f"Loaded {len(df)} rows, {df['sentence_id'].nunique()} unique sentence_ids.")
print("Rows per split:")
print(df.groupby("split")["sentence_id"].nunique())


def build_split_examples(df, split_name):
    '''Group rows for one split into a list of {sentence_id, tokens, ner_tags} dicts,
    each sorted by token_index. bio_tag strings are mapped to int ids via LABEL2ID.'''
    sub = df[df["split"] == split_name]
    examples = []
    for sid, grp in sub.groupby("sentence_id", sort=False):
        grp = grp.sort_values("token_index")
        tokens = grp["token"].tolist()
        tags = grp["bio_tag"].tolist()
        bad_tags = set(tags) - set(LABEL2ID)
        assert not bad_tags, f"Sentence {sid} has unknown bio_tag value(s): {bad_tags}"
        ner_tags = [LABEL2ID[t] for t in tags]
        # Soft BIO-consistency check (flag, don't crash -- this is a data issue, not a code bug).
        for i in range(1, len(tags)):
            if tags[i] == "I-MATH" and tags[i - 1] == "O":
                flag(f"Sentence {sid}: I-MATH at position {i} immediately follows O "
                     f"(malformed BIO span). Keeping as-is.")
                break
        examples.append({"sentence_id": sid, "tokens": tokens, "ner_tags": ner_tags})
    return examples


SPLIT_NAMES = ["test", "val", "train_clean", "train_noisy", "train_llm"]
raw = {name: build_split_examples(df, name) for name in SPLIT_NAMES}

EXPECTED_SENTENCES = {"test": 50, "val": 30, "train_clean": 185, "train_noisy": 1261, "train_llm": 510}
EXPECTED_TOKENS = {"test": 2031, "val": 1105, "train_clean": 6259, "train_noisy": 21477, "train_llm": 14550}

for name in SPLIT_NAMES:
    n_sent = len(raw[name])
    n_tok = sum(len(ex["tokens"]) for ex in raw[name])
    exp_sent, exp_tok = EXPECTED_SENTENCES[name], EXPECTED_TOKENS[name]
    status = "OK" if (n_sent == exp_sent and n_tok == exp_tok) else "MISMATCH"
    print(f"{name:12s} n_sentences={n_sent:5d} (expected {exp_sent:5d})  "
          f"n_tokens={n_tok:6d} (expected {exp_tok:6d})  [{status}]")
    if status == "MISMATCH":
        flag(f"{name}: loaded counts differ from the task-spec table. "
             f"Proceeding with what's actually in bio_merged.tsv.")

assert all(len(raw[s]) > 0 for s in SPLIT_NAMES), "One or more required splits loaded 0 sentences."

# Combined splits used by both experiments.
train_clean_plus_llm = raw["train_clean"] + raw["train_llm"]
print(f"\ntrain_clean_plus_llm: {len(train_clean_plus_llm)} sentences "
      f"({len(raw['train_clean'])} clean + {len(raw['train_llm'])} llm)")

Loaded 45422 rows, 2035 unique sentence_ids.
Rows per split:
split
test             50
train_clean     185
train_llm       509
train_noisy    1261
val              30
Name: sentence_id, dtype: int64
test         n_sentences=   50 (expected    50)  n_tokens=  2031 (expected   2031)  [OK]
val          n_sentences=   30 (expected    30)  n_tokens=  1105 (expected   1105)  [OK]
train_clean  n_sentences=  185 (expected   185)  n_tokens=  6259 (expected   6259)  [OK]
train_noisy  n_sentences= 1261 (expected  1261)  n_tokens= 21477 (expected  21477)  [OK]
train_llm    n_sentences=  509 (expected   510)  n_tokens= 14550 (expected  14550)  [MISMATCH]

[FLAG] train_llm: loaded counts differ from the task-spec table. Proceeding with what's actually in bio_merged.tsv.


train_clean_plus_llm: 694 sentences (185 clean + 509 llm)


## Tokenization + label alignment

Load the `ai-forever/ruBert-base` tokenizer (fallback `DeepPavlov/rubert-base-cased`
if the Hub load fails). Tokenize with `is_split_into_words=True`; label only the
first subword of each word, continuation subwords get `-100` (ignored by the loss
and by metrics). Sentences longer than `MAX_LENGTH=128` subwords are truncated here
(not skipped) -- consistent with how `val`/`test` must never change composition;
`train_*` sets are short transcript sentences in practice so this should rarely
trigger, but it's flagged if it does.

In [5]:
def load_tokenizer():
    global ACTIVE_MODEL_NAME
    try:
        tok = AutoTokenizer.from_pretrained(ACTIVE_MODEL_NAME)
    except Exception as e:
        flag(f"Tokenizer load for {ACTIVE_MODEL_NAME} failed: {e}. "
             f"Falling back to {FALLBACK_MODEL_NAME}.")
        ACTIVE_MODEL_NAME = FALLBACK_MODEL_NAME
        tok = AutoTokenizer.from_pretrained(ACTIVE_MODEL_NAME)
    return tok


tokenizer = load_tokenizer()
print(f"Using tokenizer for: {ACTIVE_MODEL_NAME}")


def encode_example(tokens, ner_tags, max_length=MAX_LENGTH):
    '''Tokenize one pre-split sentence and align BIO labels to first subwords.
    Returns a dict with input_ids / attention_mask / labels (python lists).'''
    enc = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=max_length)
    word_ids = enc.word_ids()
    n_subwords_untruncated = len(tokenizer(tokens, is_split_into_words=True, truncation=False)["input_ids"])
    truncated = n_subwords_untruncated > max_length
    labels, prev_word_id = [], None
    for wid in word_ids:
        if wid is None:
            labels.append(-100)
        elif wid != prev_word_id:
            labels.append(ner_tags[wid])
        else:
            labels.append(-100)
        prev_word_id = wid
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": labels,
    }, truncated


def encode_split(examples, name):
    encoded = []
    n_truncated = 0
    for ex in examples:
        enc, truncated = encode_example(ex["tokens"], ex["ner_tags"])
        if truncated:
            n_truncated += 1
        encoded.append(enc)
    if n_truncated > 0:
        flag(f"{name}: {n_truncated}/{len(examples)} sentence(s) exceeded {MAX_LENGTH} subwords "
             f"and were truncated (not skipped).")
    return encoded


encoded = {
    "test": encode_split(raw["test"], "test"),
    "val": encode_split(raw["val"], "val"),
    "train_noisy": encode_split(raw["train_noisy"], "train_noisy"),
    "train_clean_plus_llm": encode_split(train_clean_plus_llm, "train_clean_plus_llm"),
}

# --- Alignment sanity check on a few examples ---
print("Alignment check (first 3 examples of train_clean_plus_llm):")
for i in range(min(3, len(train_clean_plus_llm))):
    orig = train_clean_plus_llm[i]
    subwords = tokenizer.convert_ids_to_tokens(encoded["train_clean_plus_llm"][i]["input_ids"])
    labs = encoded["train_clean_plus_llm"][i]["labels"]
    pairs = [(sw, ID2LABEL.get(l, "IGN")) for sw, l in zip(subwords, labs)]
    print(f"  ex {i}: {pairs}")
    non_ignored = [l for l in labs if l != -100]
    assert len(non_ignored) == len(orig["ner_tags"]) or len(non_ignored) < len(orig["ner_tags"]), (
        "Alignment produced MORE labeled subtokens than original words -- bug in encode_example."
    )
print("Alignment check passed: at most one labeled subtoken per original word "
      "(fewer only if a sentence was truncated, which would have been flagged above).")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

Using tokenizer for: ai-forever/ruBert-base

[FLAG] test: 2/50 sentence(s) exceeded 128 subwords and were truncated (not skipped).

Alignment check (first 3 examples of train_clean_plus_llm):
  ex 0: [('[CLS]', 'IGN'), ('по', 'O'), ('свои', 'O'), ('##ству', 'IGN'), ('логариф', 'O'), ('##мов', 'IGN'), (',', 'IGN'), ('логариф', 'B-MATH'), ('##м', 'IGN'), ('произведения', 'I-MATH'), ('экспонен', 'I-MATH'), ('##ты', 'IGN'), ('в', 'I-MATH'), ('степени', 'I-MATH'), ('квадрата', 'I-MATH'), ('логариф', 'I-MATH'), ('##ма', 'IGN'), ('икс', 'I-MATH'), ('и', 'I-MATH'), ('игре', 'I-MATH'), ('##к', 'IGN'), ('распадается', 'O'), ('в', 'O'), ('сумму', 'O'), (',', 'IGN'), ('а', 'O'), ('логариф', 'B-MATH'), ('##м', 'IGN'), ('от', 'I-MATH'), ('произведения', 'I-MATH'), ('а', 'I-MATH'), ('и', 'I-MATH'), ('бэ', 'I-MATH'), ('равен', 'O'), ('сумме', 'O'), ('логариф', 'O'), ('##мов', 'IGN'), ('сом', 'O'), ('##но', 'IGN'), ('##жите', 'IGN'), ('##леи', 'IGN'), ('.', 'IGN'), ('[SEP]', 'IGN')]
  ex 1: [('[CLS]', 

## Dataset + dynamic-padding DataLoader

No `Trainer`/`DataCollatorForTokenClassification` here -- a small `Dataset` wrapper
plus a hand-written `collate_fn` that pads `input_ids`/`attention_mask` (with the
tokenizer's pad id) and `labels` (with `-100`) to the longest sequence in each batch.

In [6]:
class BIODataset(Dataset):
    def __init__(self, encoded_examples):
        self.data = encoded_examples

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0


def collate_fn(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        pad_len = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [PAD_TOKEN_ID] * pad_len)
        attention_mask.append(ex["attention_mask"] + [0] * pad_len)
        labels.append(ex["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


def make_loader(encoded_examples, batch_size, shuffle):
    return DataLoader(
        BIODataset(encoded_examples),
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_fn,
    )

## Metrics

- **Primary: span-level seqeval F1**, strict mode, IOB2 scheme (exact boundary match).
- **Secondary: token-level F1**, binary (B-MATH/I-MATH vs O).
- **Exact match**: fraction of sentences where every token's predicted tag matches gold.

All computed on first-subword positions only (the same positions the loss is
computed on); `-100` positions are skipped when decoding.

In [7]:
def decode_batch(logits, labels):
    '''logits: (B, T, NUM_LABELS) tensor: labels: (B, T) tensor with -100 at ignored
    positions. Returns (true_seqs, pred_seqs) lists of string-tag lists, one per
    sentence in the batch, skipping -100 positions.'''
    pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
    label_ids = labels.cpu().numpy()
    true_seqs, pred_seqs = [], []
    for p_row, l_row in zip(pred_ids, label_ids):
        true_seq, pred_seq = [], []
        for p, l in zip(p_row, l_row):
            if l == -100:
                continue
            true_seq.append(ID2LABEL[int(l)])
            pred_seq.append(ID2LABEL[int(p)])
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)
    return true_seqs, pred_seqs


def compute_metrics(true_seqs, pred_seqs):
    span_f1 = seqeval_f1(true_seqs, pred_seqs, mode="strict", scheme=IOB2)

    flat_true = [t for seq in true_seqs for t in seq]
    flat_pred = [t for seq in pred_seqs for t in seq]
    bin_true = [0 if t == "O" else 1 for t in flat_true]
    bin_pred = [0 if t == "O" else 1 for t in flat_pred]
    token_f1 = sklearn_f1(bin_true, bin_pred, average="binary", zero_division=0)

    exact_matches = sum(1 for t, p in zip(true_seqs, pred_seqs) if t == p)
    exact_match = exact_matches / len(true_seqs) if true_seqs else 0.0

    return {"span_f1": span_f1, "token_f1": token_f1, "exact_match": exact_match}


@torch.no_grad()
def evaluate_split(model, loader):
    '''Runs the eval loop, accumulates per-sentence tag sequences across all
    batches, and returns (metrics_dict, true_seqs, pred_seqs). true_seqs/pred_seqs
    are kept so a future error-analysis pass can use them without re-running the
    model.'''
    model.eval()
    all_true, all_pred = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        true_seqs, pred_seqs = decode_batch(outputs.logits, labels)
        all_true.extend(true_seqs)
        all_pred.extend(pred_seqs)
    metrics = compute_metrics(all_true, all_pred)
    return metrics, all_true, all_pred

## Manual training loop

No `Trainer`. `AdamW` + linear warmup (10% of total steps) + gradient clipping
(`max_norm=1.0`) + manual fp16 (`autocast`/`GradScaler`, skipped automatically on
CPU). Best-checkpoint tracking is in-memory only (`BestCheckpointTracker` clones
`model.state_dict()` to CPU on improvement and reloads it at the end) -- no
per-epoch disk writes. A fresh `AdamW`/scheduler is created on every
`train_one_stage` call, which is what gives BERT-5's Stage 2 a genuinely fresh
optimizer while keeping the same (already Stage-1-trained) in-memory model --
no checkpoint save/reload round-trip needed for that handoff.

OOM handling: if a stage hits a CUDA OOM, it's caught, flagged, and retried once
with `batch_size=8` + `gradient_accumulation_steps=2` (same effective batch size).

In [8]:
class BestCheckpointTracker:
    '''Keeps the best-eval state_dict in CPU memory (no disk I/O) and reports
    whether `patience` evals have passed with no improvement on span-F1.'''

    def __init__(self, model, patience):
        self.model = model
        self.patience = patience
        self.best_metric = None
        self.best_epoch = None
        self.best_state = None
        self.num_bad = 0

    def step(self, epoch, value):
        if self.best_metric is None or value > self.best_metric:
            self.best_metric = value
            self.best_epoch = epoch
            self.best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
            self.num_bad = 0
            return False
        self.num_bad += 1
        return self.patience is not None and self.num_bad >= self.patience

    def restore_best(self):
        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
            self.model.to(DEVICE)
        return self.best_epoch, self.best_metric


def build_model():
    '''Fresh AutoModelForTokenClassification from ACTIVE_MODEL_NAME, with a
    one-time fallback to FALLBACK_MODEL_NAME if the Hub load fails.
    classifier_dropout=0.3 per the project's common ruBERT settings; if a given
    model's config doesn't accept that kwarg, retry without it.'''
    global ACTIVE_MODEL_NAME

    def _load(name):
        try:
            return AutoModelForTokenClassification.from_pretrained(
                name, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
                classifier_dropout=0.3,
            )
        except TypeError:
            return AutoModelForTokenClassification.from_pretrained(
                name, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
            )

    try:
        model = _load(ACTIVE_MODEL_NAME)
    except Exception as e:
        if ACTIVE_MODEL_NAME != FALLBACK_MODEL_NAME:
            flag(f"Model load for {ACTIVE_MODEL_NAME} failed: {e}. Falling back to {FALLBACK_MODEL_NAME}.")
            ACTIVE_MODEL_NAME = FALLBACK_MODEL_NAME
            model = _load(ACTIVE_MODEL_NAME)
        else:
            raise
    return model.to(DEVICE)


def train_one_stage(
    model,
    train_encoded,
    eval_encoded,
    lr,
    max_epochs,
    batch_size=16,
    early_stopping_patience=None,
    label_smoothing=0.0,
    seed=42,
    run_label="run",
):
    '''Train one stage with a manual loop. Returns (model, elapsed_minutes,
    best_epoch, best_val_span_f1). best_epoch/best_val_span_f1 are None when
    eval_encoded/early_stopping_patience aren't both set (e.g. BERT-5 Stage 1,
    which trains a fixed number of epochs with no val signal).'''
    do_eval = eval_encoded is not None and early_stopping_patience is not None
    use_amp = (DEVICE == "cuda")

    bsz, grad_accum = batch_size, 1
    attempt = 0
    start = time.time()
    tracker = None
    span_f1_history = []

    while True:
        attempt += 1
        try:
            set_all_seeds(seed)
            train_loader = make_loader(train_encoded, bsz, shuffle=True)
            eval_loader = make_loader(eval_encoded, 32, shuffle=False) if do_eval else None

            optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
            steps_per_epoch = max(1, len(train_loader) // grad_accum)
            total_steps = steps_per_epoch * max_epochs
            warmup_steps = max(1, int(0.1 * total_steps))
            scheduler = get_linear_schedule_with_warmup(
                optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
            )
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=label_smoothing)
            scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
            tracker = BestCheckpointTracker(model, patience=early_stopping_patience) if do_eval else None
            span_f1_history = []

            for epoch in range(1, max_epochs + 1):
                model.train()
                running_loss, n_steps = 0.0, 0
                optimizer.zero_grad()
                for step, batch in enumerate(train_loader):
                    input_ids = batch["input_ids"].to(DEVICE)
                    attention_mask = batch["attention_mask"].to(DEVICE)
                    labels = batch["labels"].to(DEVICE)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                        loss = loss_fn(outputs.logits.view(-1, NUM_LABELS), labels.view(-1))
                        loss = loss / grad_accum
                    scaler.scale(loss).backward()
                    running_loss += loss.item() * grad_accum
                    n_steps += 1
                    is_last_batch = (step + 1) == len(train_loader)
                    if (step + 1) % grad_accum == 0 or is_last_batch:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        scaler.step(optimizer)
                        scaler.update()
                        scheduler.step()
                        optimizer.zero_grad()

                avg_loss = running_loss / max(1, n_steps)
                msg = f"[{run_label}] epoch {epoch}/{max_epochs}  train_loss={avg_loss:.4f}"

                if do_eval:
                    val_metrics, _, _ = evaluate_split(model, eval_loader)
                    span_f1_history.append(val_metrics["span_f1"])
                    msg += f"  val_span_f1={val_metrics['span_f1']:.4f}"
                    print(msg)
                    should_stop = tracker.step(epoch, val_metrics["span_f1"])
                    if should_stop:
                        print(f"[{run_label}] early stopping at epoch {epoch} "
                              f"(no improvement for {early_stopping_patience} epochs).")
                        break
                else:
                    print(msg)
            break  # stage completed without OOM
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            torch.cuda.empty_cache()
            gc.collect()
            if attempt >= 2:
                raise
            flag(f"[{run_label}] CUDA OOM with batch_size={bsz}. Retrying with "
                 f"batch_size=8, gradient_accumulation_steps=2 (effective batch size unchanged).")
            bsz, grad_accum = 8, 2

    elapsed_min = (time.time() - start) / 60.0

    if do_eval and len(span_f1_history) > 0 and all(v == 0.0 for v in span_f1_history):
        raise RuntimeError(
            f"[{run_label}] val span-F1 is exactly 0.0 for ALL epochs. This indicates a "
            f"label-alignment or metric-computation bug. Stopping per protocol instead of "
            f"continuing -- inspect encode_example / compute_metrics before re-running."
        )

    best_epoch, best_val_f1 = (None, None)
    if tracker is not None:
        best_epoch, best_val_f1 = tracker.restore_best()  # loads best-eval weights back into `model` in place

    return model, elapsed_min, best_epoch, best_val_f1

## BERT-4: Clean (185) + LLM (510) = 695 sentences, no curriculum

LR 2e-5, max 10 epochs, early stopping patience=3 on val span-F1. 3 seeds.

In [9]:
bert4_results = []
bert4_best_overall = {"seed": None, "epoch": None, "val_span_f1": -1.0}
bert4_test_predictions = {}  # seed -> (true_seqs, pred_seqs)

for seed in SEEDS:
    print(f"\n=== BERT-4 | seed={seed} ===")
    set_all_seeds(seed)
    model = build_model()

    model, elapsed_min, best_epoch, best_val_f1 = train_one_stage(
        model=model,
        train_encoded=encoded["train_clean_plus_llm"],
        eval_encoded=encoded["val"],
        lr=2e-5,
        max_epochs=10,
        batch_size=16,
        early_stopping_patience=3,
        label_smoothing=0.0,
        seed=seed,
        run_label=f"BERT-4 seed{seed}",
    )

    val_metrics, _, _ = evaluate_split(model, make_loader(encoded["val"], 32, shuffle=False))
    test_loader = make_loader(encoded["test"], 32, shuffle=False)
    test_metrics, true_seqs, pred_seqs = evaluate_split(model, test_loader)
    bert4_test_predictions[seed] = (true_seqs, pred_seqs)

    if best_val_f1 is not None and best_val_f1 > bert4_best_overall["val_span_f1"]:
        bert4_best_overall = {"seed": seed, "epoch": best_epoch, "val_span_f1": best_val_f1}

    bert4_results.append({
        "seed": seed,
        "val_span_f1": val_metrics["span_f1"],
        "test_span_f1": test_metrics["span_f1"],
        "test_token_f1": test_metrics["token_f1"],
        "test_exact_match": test_metrics["exact_match"],
        "train_time_min": elapsed_min,
    })
    print(bert4_results[-1])

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

bert4_total_min = sum(r["train_time_min"] for r in bert4_results)
if bert4_total_min > 120:
    flag(f"BERT-4 total training time across {len(SEEDS)} seeds was {bert4_total_min:.1f} min "
         f"(> 2h). Consider batch_size=8, fewer max epochs, or gradient accumulation if this "
         f"recurs.")
print(f"\nBERT-4 total training time across {len(SEEDS)} seeds: {bert4_total_min:.1f} min")


=== BERT-4 | seed=42 ===


pytorch_model.bin:   0%|          | 0.00/716M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

model.safetensors:   0%|          | 0.00/716M [00:00<?, ?B/s]

...

[BERT-4 seed456] epoch 4/10  train_loss=0.1332  val_span_f1=0.2772


/tmp/ipykernel_977/869366990.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[BERT-4 seed456] epoch 5/10  train_loss=0.0999  val_span_f1=0.2673


/tmp/ipykernel_977/869366990.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[BERT-4 seed456] epoch 6/10  train_loss=0.0727  val_span_f1=0.2557
[BERT-4 seed456] early stopping at epoch 6 (no improvement for 3 epochs).
{'seed': 456, 'val_span_f1': np.float64(0.28421052631578947), 'test_span_f1': np.float64(0.30128205128205127), 'test_token_f1': 0.8465298142717498, 'test_exact_match': 0.08, 'train_time_min': 0.6222164591153463}

BERT-4 total training time across 3 seeds: 2.0 min


## BERT-5: Curriculum -- Stage 1 (Mathematicon) -> Stage 2 (Clean + LLM)

Stage 1: `train_noisy` (1261 sentences), LR 3e-5, 3 fixed epochs, label_smoothing=0.1,
no eval/early stopping. Stage 2: fresh optimizer (same in-memory, already-Stage-1-trained
model), `train_clean_plus_llm` (695), LR 1.5e-5, max 15 epochs, early stopping
patience=3 on val span-F1. 3 seeds.

In [10]:
bert5_results = []
bert5_best_overall = {"seed": None, "epoch": None, "val_span_f1": -1.0}
bert5_test_predictions = {}  # seed -> (true_seqs, pred_seqs)

for seed in SEEDS:
    print(f"\n=== BERT-5 | seed={seed} | Stage 1 (noisy) ===")
    set_all_seeds(seed)
    model = build_model()

    model, stage1_min, _, _ = train_one_stage(
        model=model,
        train_encoded=encoded["train_noisy"],
        eval_encoded=None,
        lr=3e-5,
        max_epochs=3,
        batch_size=16,
        early_stopping_patience=None,
        label_smoothing=0.1,
        seed=seed,
        run_label=f"BERT-5 seed{seed} stage1",
    )

    print(f"=== BERT-5 | seed={seed} | Stage 2 (clean+llm, fresh optimizer) ===")
    # Same in-memory model (carries Stage-1 weights forward); train_one_stage builds
    # a brand-new AdamW + scheduler on this call, so the optimizer is genuinely fresh
    # with no disk save/reload round-trip needed for the handoff.
    model, stage2_min, best_epoch, best_val_f1 = train_one_stage(
        model=model,
        train_encoded=encoded["train_clean_plus_llm"],
        eval_encoded=encoded["val"],
        lr=1.5e-5,
        max_epochs=15,
        batch_size=16,
        early_stopping_patience=3,
        label_smoothing=0.0,
        seed=seed,
        run_label=f"BERT-5 seed{seed} stage2",
    )

    val_metrics, _, _ = evaluate_split(model, make_loader(encoded["val"], 32, shuffle=False))
    test_loader = make_loader(encoded["test"], 32, shuffle=False)
    test_metrics, true_seqs, pred_seqs = evaluate_split(model, test_loader)
    bert5_test_predictions[seed] = (true_seqs, pred_seqs)

    if best_val_f1 is not None and best_val_f1 > bert5_best_overall["val_span_f1"]:
        bert5_best_overall = {"seed": seed, "epoch": best_epoch, "val_span_f1": best_val_f1}

    total_min = stage1_min + stage2_min
    bert5_results.append({
        "seed": seed,
        "val_span_f1": val_metrics["span_f1"],
        "test_span_f1": test_metrics["span_f1"],
        "test_token_f1": test_metrics["token_f1"],
        "test_exact_match": test_metrics["exact_match"],
        "stage1_time_min": stage1_min,
        "stage2_time_min": stage2_min,
        "train_time_min": total_min,
    })
    print(bert5_results[-1])

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

bert5_total_min = sum(r["train_time_min"] for r in bert5_results)
if bert5_total_min > 120:
    flag(f"BERT-5 total training time across {len(SEEDS)} seeds was {bert5_total_min:.1f} min "
         f"(> 2h). Consider batch_size=8, fewer max epochs, or gradient accumulation if this "
         f"recurs.")
print(f"\nBERT-5 total training time across {len(SEEDS)} seeds: {bert5_total_min:.1f} min")


=== BERT-5 | seed=42 | Stage 1 (noisy) ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

[BERT-5 seed42 stage1] epoch 1/3  train_loss=0.7631
[BERT-5 seed42 stage1] epoch 2/3  train_loss=0.4960
[BERT-5 seed42 stage1] epoch 3/3  train_loss=0.4058
=== BERT-5 | seed=42 | Stage 2 (clean+llm, fresh optimizer) ===
[BERT-5 seed42 stage2] epoch 1/15  train_loss=0.7641  val_span_f1=0.0339


/tmp/ipykernel_977/869366990.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


...

[BERT-5 seed456 stage2] epoch 6/15  train_loss=0.1173  val_span_f1=0.2618


/tmp/ipykernel_977/869366990.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[BERT-5 seed456 stage2] epoch 7/15  train_loss=0.0915  val_span_f1=0.2741


/tmp/ipykernel_977/869366990.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[BERT-5 seed456 stage2] epoch 8/15  train_loss=0.0732  val_span_f1=0.2788
[BERT-5 seed456 stage2] early stopping at epoch 8 (no improvement for 3 epochs).
{'seed': 456, 'val_span_f1': np.float64(0.2803738317757009), 'test_span_f1': np.float64(0.3295454545454546), 'test_token_f1': 0.8521229868228404, 'test_exact_match': 0.06, 'stage1_time_min': 0.48077261050542197, 'stage2_time_min': 0.8370035330454508, 'train_time_min': 1.3177761435508728}

BERT-5 total training time across 3 seeds: 3.8 min


## Results summary

Builds and prints the `<executor_output id="E4d">` block: per-seed tables +
mean±std for BERT-4 and BERT-5, best-checkpoint / training-time lines, the full
BERT-0..BERT-5 comparison table (BERT-0..BERT-3 rows are hardcoded from prior
results per the task spec; BERT-4/BERT-5 rows are computed from the runs above),
and a one-line Notes summary.

In [11]:
def mean_std(values):
    arr = np.array(values, dtype=float)
    return float(arr.mean()), float(arr.std())


def fmt(v, nd=4):
    return f"{v:.{nd}f}"


def render_results_table(results):
    lines = ["| Seed | Val Span-F1 | Test Span-F1 | Test Token-F1 | Test Exact Match |",
             "|------|-------------|--------------|---------------|------------------|"]
    for r in results:
        lines.append(
            f"| {r['seed']}   | {fmt(r['val_span_f1'])}      | {fmt(r['test_span_f1'])}       "
            f"| {fmt(r['test_token_f1'])}        | {fmt(r['test_exact_match'])}           |"
        )
    val_mean, val_std = mean_std([r["val_span_f1"] for r in results])
    test_span_mean, test_span_std = mean_std([r["test_span_f1"] for r in results])
    test_tok_mean, test_tok_std = mean_std([r["test_token_f1"] for r in results])
    test_em_mean, test_em_std = mean_std([r["test_exact_match"] for r in results])
    lines.append(
        f"| Mean±Std | {fmt(val_mean)}±{fmt(val_std)} | {fmt(test_span_mean)}±{fmt(test_span_std)} "
        f"| {fmt(test_tok_mean)}±{fmt(test_tok_std)} | {fmt(test_em_mean)}±{fmt(test_em_std)} |"
    )
    return "\n".join(lines), test_span_mean, test_span_std


bert4_table, bert4_test_span_mean, bert4_test_span_std = render_results_table(bert4_results)
bert5_table, bert5_test_span_mean, bert5_test_span_std = render_results_table(bert5_results)

BERT0_MEAN, BERT0_STD = 0.1932, 0.0043
BERT1_MEAN, BERT1_STD = 0.1844, 0.0177
BERT2_MEAN, BERT2_STD = 0.1634, 0.0028
BERT3_MEAN, BERT3_STD = 0.1509, 0.0152

bert4_delta = bert4_test_span_mean - BERT0_MEAN
bert5_delta = bert5_test_span_mean - BERT0_MEAN

bert4_tok_mean, _ = mean_std([r["test_token_f1"] for r in bert4_results])
bert5_tok_mean, _ = mean_std([r["test_token_f1"] for r in bert5_results])

llm_helped = bert4_delta > 0
curriculum_still_hurts = bert5_test_span_mean < bert4_test_span_mean

notes = (
    f"LLM-augmented data {'improved' if llm_helped else 'did not improve'} span-F1 vs BERT-0 "
    f"(BERT-4 {fmt(bert4_test_span_mean)} vs {fmt(BERT0_MEAN)}, "
    f"{'+' if bert4_delta >= 0 else ''}{fmt(bert4_delta)}); "
    f"curriculum {'still hurt' if curriculum_still_hurts else 'did not hurt'} relative to no-curriculum "
    f"(BERT-5 {fmt(bert5_test_span_mean)} vs BERT-4 {fmt(bert4_test_span_mean)}); "
    f"token-F1 stayed high (BERT-4 {fmt(bert4_tok_mean)}, BERT-5 {fmt(bert5_tok_mean)}) "
    f"while span-F1 remained much lower, so boundary detection is still the bottleneck."
)

output_block = f'''<executor_output id="E4d">
## ruBERT v3 Experiment Results (LLM-generated data)

### BERT-4: Clean (185) + LLM (510) = 695 sentences
{bert4_table}
Best checkpoint: seed={bert4_best_overall['seed']}, epoch={bert4_best_overall['epoch']}, val_span_f1={fmt(bert4_best_overall['val_span_f1'])}
Training time per seed (approx): {fmt(bert4_total_min / len(SEEDS), 1)} min

### BERT-5: Curriculum (Mathematicon -> Clean + LLM)
{bert5_table}
Best checkpoint: seed={bert5_best_overall['seed']}, epoch={bert5_best_overall['epoch']}, val_span_f1={fmt(bert5_best_overall['val_span_f1'])}
Training time per seed (approx): {fmt(bert5_total_min / len(SEEDS), 1)} min (stage1+stage2)

### Full Comparison (include previous results for context)
| Model | Config | Train size | Test Span-F1 (mean±std) | Δ vs BERT-0 |
|-------|--------|-----------|------------------------|-------------|
| BERT-0 | clean-only | 185 | {fmt(BERT0_MEAN)}±{fmt(BERT0_STD)} | — |
| BERT-1 | curriculum Math->clean | 1261+185 | {fmt(BERT1_MEAN)}±{fmt(BERT1_STD)} | {fmt(BERT1_MEAN - BERT0_MEAN)} |
| BERT-2 | clean + s2l_aug | 185+2000 | {fmt(BERT2_MEAN)}±{fmt(BERT2_STD)} | {fmt(BERT2_MEAN - BERT0_MEAN)} |
| BERT-3 | curriculum + s2l | 1261+185+2000 | {fmt(BERT3_MEAN)}±{fmt(BERT3_STD)} | {fmt(BERT3_MEAN - BERT0_MEAN)} |
| BERT-4 | clean + LLM | 185+510 | {fmt(bert4_test_span_mean)}±{fmt(bert4_test_span_std)} | {'+' if bert4_delta >= 0 else ''}{fmt(bert4_delta)} |
| BERT-5 | curriculum -> clean+LLM | 1261+695 | {fmt(bert5_test_span_mean)}±{fmt(bert5_test_span_std)} | {'+' if bert5_delta >= 0 else ''}{fmt(bert5_delta)} |

### Notes
{notes}
</executor_output>'''

print(output_block)

<executor_output id="E4d">
## ruBERT v3 Experiment Results (LLM-generated data)

### BERT-4: Clean (185) + LLM (510) = 695 sentences
| Seed | Val Span-F1 | Test Span-F1 | Test Token-F1 | Test Exact Match |
|------|-------------|--------------|---------------|------------------|
| 42   | 0.2759      | 0.3333       | 0.8416        | 0.0800           |
| 123   | 0.2474      | 0.3514       | 0.8493        | 0.1000           |
| 456   | 0.2842      | 0.3013       | 0.8465        | 0.0800           |
| Mean±Std | 0.2692±0.0157 | 0.3287±0.0207 | 0.8458±0.0032 | 0.0867±0.0094 |
Best checkpoint: seed=456, epoch=3, val_span_f1=0.2842
Training time per seed (approx): 0.7 min

### BERT-5: Curriculum (Mathematicon -> Clean + LLM)
| Seed | Val Span-F1 | Test Span-F1 | Test Token-F1 | Test Exact Match |
|------|-------------|--------------|---------------|------------------|
| 42   | 0.2944      | 0.2838       | 0.8222        | 0.1000           |
| 123   | 0.3125      | 0.3081       | 0.8448        |

In [12]:
def extract_spans(tags):
    """Contiguous B-MATH(I-MATH)* runs -> list of (start, end_exclusive)."""
    spans = []
    start = None
    for i, t in enumerate(tags):
        if t == "B-MATH":
            if start is not None:
                spans.append((start, i))
            start = i
        elif t == "I-MATH":
            if start is None:
                start = i  # malformed (I-MATH without a preceding B-MATH); defensive only
        else:
            if start is not None:
                spans.append((start, i))
                start = None
    if start is not None:
        spans.append((start, len(tags)))
    return spans


def render_sentence(tokens, tags):
    return " ".join(f"{tok}/{tag}" if tag != "O" else tok for tok, tag in zip(tokens, tags))


def span_text(tokens, span):
    s, e = span
    return " ".join(tokens[s:e])


def print_error_analysis(model_name, true_seqs, pred_seqs, tokens_list):
    print(f"\n{'='*90}\n{model_name}: test-set predictions ({len(true_seqs)} sentences)\n{'='*90}")

    n_exact = 0
    error_rows = []
    for i, (full_tokens, gold, pred) in enumerate(zip(tokens_list, true_seqs, pred_seqs)):
        # Align raw tokens to the (possibly truncated, first-subword-only) label
        # sequence: truncation only ever drops trailing words, so slicing to
        # len(gold) keeps the correspondence correct even for long sentences.
        tokens = full_tokens[:len(gold)]
        assert len(tokens) == len(gold) == len(pred), (
            f"Length mismatch at test sentence {i}: tokens={len(tokens)} gold={len(gold)} pred={len(pred)}"
        )
        if len(full_tokens) != len(gold):
            flag(f"{model_name}: test sentence {i} was truncated for evaluation "
                 f"({len(full_tokens)} raw tokens -> {len(gold)} labeled); "
                 f"showing only the retained prefix below.")
        is_exact = gold == pred
        if is_exact:
            n_exact += 1
        print(f"\n[{i}] GOLD: {render_sentence(tokens, gold)}")
        print(f"[{i}] PRED: {render_sentence(tokens, pred)}")
        if not is_exact:
            gold_spans = extract_spans(gold)
            pred_spans = extract_spans(pred)
            gold_set, pred_set = set(gold_spans), set(pred_spans)
            missed = gold_set - pred_set    # gold spans with no exact predicted match
            extra = pred_set - gold_set     # predicted spans with no exact gold match
            error_rows.append((i, tokens, gold_spans, pred_spans, missed, extra))
            print(f"[{i}] MISMATCH -- gold: {[span_text(tokens, s) for s in gold_spans]} "
                  f"| pred: {[span_text(tokens, s) for s in pred_spans]}")

    n_total = len(true_seqs)
    n_missed_total = sum(len(r[4]) for r in error_rows)
    n_extra_total = sum(len(r[5]) for r in error_rows)

    print(f"\n--- {model_name}: errors-only summary ---")
    print(f"Exact-match sentences: {n_exact}/{n_total} ({n_exact/n_total:.4f})")
    print(f"Sentences with >=1 span error: {len(error_rows)}/{n_total}")
    print(f"Missed gold spans (false negatives, no exact predicted match): {n_missed_total}")
    print(f"Extra predicted spans (false positives, no exact gold match): {n_extra_total}")

    print(f"\n--- {model_name}: detailed error list ---")
    if not error_rows:
        print("(no errors -- every test sentence matched exactly)")
    for i, tokens, gold_spans, pred_spans, missed, extra in error_rows:
        print(f"\n[{i}] sentence: {' '.join(tokens)}")
        print(f"    gold spans:      {[span_text(tokens, s) for s in gold_spans]}")
        print(f"    predicted spans: {[span_text(tokens, s) for s in pred_spans]}")
        if missed:
            print(f"    MISSED (gold, no exact prediction):  {[span_text(tokens, s) for s in missed]}")
        if extra:
            print(f"    EXTRA (predicted, not in gold):      {[span_text(tokens, s) for s in extra]}")

    return {"n_exact": n_exact, "n_total": n_total, "n_missed": n_missed_total, "n_extra": n_extra_total}


test_tokens = [ex["tokens"] for ex in raw["test"]]

bert4_err_seed = bert4_best_overall["seed"]
bert5_err_seed = bert5_best_overall["seed"]

if bert4_err_seed in bert4_test_predictions:
    true4, pred4 = bert4_test_predictions[bert4_err_seed]
    bert4_error_summary = print_error_analysis(
        f"BERT-4 (best seed={bert4_err_seed})", true4, pred4, test_tokens
    )
else:
    flag(f"No stored test predictions for BERT-4 seed {bert4_err_seed}; skipping its error analysis.")

if bert5_err_seed in bert5_test_predictions:
    true5, pred5 = bert5_test_predictions[bert5_err_seed]
    bert5_error_summary = print_error_analysis(
        f"BERT-5 (best seed={bert5_err_seed})", true5, pred5, test_tokens
    )
else:
    flag(f"No stored test predictions for BERT-5 seed {bert5_err_seed}; skipping its error analysis.")


BERT-4 (best seed=456): test-set predictions (50 sentences)

[FLAG] BERT-4 (best seed=456): test sentence 0 was truncated for evaluation (112 raw tokens -> 104 labeled); showing only the retained prefix below.


[0] GOLD: Итак, число B называется пределом/B-MATH функций/I-MATH х/I-MATH от/I-MATH двух/I-MATH переменных/I-MATH х/I-MATH и/I-MATH грек/I-MATH в/I-MATH точке/I-MATH A/I-MATH . Если существует/B-MATH такое/I-MATH дельта/I-MATH , только все-таки начнем с другого. Если/B-MATH для/I-MATH любого/I-MATH эпселен/I-MATH большего/I-MATH нуля,/I-MATH существует/I-MATH такое/I-MATH дельта/I-MATH больше/I-MATH нуля,/I-MATH то/I-MATH для/I-MATH всех/I-MATH точек/I-MATH M/I-MATH ,/I-MATH принадлежащих/I-MATH области/I-MATH определения, это очень важно. Точно так же, как и в случае одной перемены. Мы там тоже говорили о том, что должно принадлежать области определения. Таких,/B-MATH что/I-MATH расстояние/I-MATH от/I-MATH точки/I-MATH M/I-MATH до/I-MATH точки/I-MATH A/I-MATH меньше/I-MATH д